In [20]:
from google.colab import drive
drive.mount('/content/drive')
import os
import json
import shutil
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
print("FEATURE ENGINEERING & SELECTION")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FEATURE ENGINEERING & SELECTION


In [21]:
day12_output_directory = ( "/content/drive/MyDrive/""BATTERY_SOC_PROJECT/day12_dataset_V1")
day13_output_directory = ("/content/drive/MyDrive/""BATTERY_SOC_PROJECT/day13_feature_engineering")
os.makedirs(day13_output_directory, exist_ok=True)
print("Day 12 input folder:")
print(day12_output_directory)
print("\nDay 13 output folder:")
print(day13_output_directory)

Day 12 input folder:
/content/drive/MyDrive/BATTERY_SOC_PROJECT/day12_dataset_V1

Day 13 output folder:
/content/drive/MyDrive/BATTERY_SOC_PROJECT/day13_feature_engineering


In [22]:
soc_train = pd.read_csv(
os.path.join(day12_output_directory,"SOC_train_day12_V1.csv"))
soc_validation = pd.read_csv(os.path.join(day12_output_directory,"SOC_validation_day12_V1.csv"))
soc_test = pd.read_csv( os.path.join( day12_output_directory,"SOC_test_day12_V1.csv" ))
time_train = pd.read_csv(os.path.join(day12_output_directory,"remaining_time_train_day12_V1.csv"))
time_validation = pd.read_csv(os.path.join(day12_output_directory,"remaining_time_validation_day12_V1.csv"))
time_test = pd.read_csv(os.path.join(day12_output_directory,"remaining_time_test_day12_V1.csv"))
print("DAY 12 DATASET V1 LOADED")
print("\nSOC datasets:")
print("Train:", soc_train.shape)
print("Validation:", soc_validation.shape)
print("Test:", soc_test.shape)
print("\nRemaining-time datasets:")
print("Train:", time_train.shape)
print("Validation:", time_validation.shape)
print("Test:", time_test.shape)

DAY 12 DATASET V1 LOADED

SOC datasets:
Train: (9926, 9)
Validation: (7376, 9)
Test: (2250, 9)

Remaining-time datasets:
Train: (41015, 10)
Validation: (1349, 10)
Test: (2286, 10)


In [23]:
soc_raw_features = ["Voltage","Current","Temperature","Elapsed Time","Cycle"]
time_raw_features = ["Voltage","Current","Temperature", "Elapsed Time","Cycle","Capacity"]
if "WhAccu" in soc_train.columns:
    soc_raw_features.append("WhAccu")
if "WhAccu" in time_train.columns:
    time_raw_features.append("WhAccu")
soc_raw_features = [
    col for col in soc_raw_features
    if col in soc_train.columns]
time_raw_features = [
    col for col in time_raw_features
    if col in time_train.columns]
print("SOC raw features:")
print(soc_raw_features)
print("\nRemaining-time raw features:")
print(time_raw_features)

SOC raw features:
['Voltage', 'Current', 'Temperature', 'Elapsed Time', 'Cycle', 'WhAccu']

Remaining-time raw features:
['Voltage', 'Current', 'Temperature', 'Elapsed Time', 'Cycle', 'Capacity', 'WhAccu']


In [24]:
for df in [soc_train,soc_validation,soc_test,time_train,time_validation,time_test]:
    df["Power"] = df["Voltage"] * df["Current"]
print("\n Power feature created.")
print("\nPower statistics:")
print(soc_train["Power"].describe())
print("\nSample values:")
print(soc_train[["Voltage", "Current", "Power"]].head())


 Power feature created.

Power statistics:
count    9926.000000
mean        3.853999
std         4.674849
min         0.203816
25%         0.541205
50%         0.607972
75%         8.506297
max        12.592177
Name: Power, dtype: float64

Sample values:
   Voltage  Current     Power
0  3.01141  0.15069  0.453789
1  3.03281  0.14814  0.449280
2  3.05169  0.14814  0.452077
3  3.06872  0.14814  0.454600
4  3.08456  0.14814  0.456947


In [25]:
print("\nMissing Power values:")
print("SOC:",soc_train["Power"].isna().sum())
print("Remaining Time:",time_train["Power"].isna().sum())
print("\nInfinite Power values:")
print("SOC:", np.isinf(soc_train["Power"]).sum())
print("Remaining Time:",np.isinf(time_train["Power"]).sum())
print("\nPower range:")
print("Minimum:",soc_train["Power"].min())
print("Maximum:",soc_train["Power"].max())
print("\nPower feature check completed.")


Missing Power values:
SOC: 0
Remaining Time: 0

Infinite Power values:
SOC: 0
Remaining Time: 0

Power range:
Minimum: 0.2038158087
Maximum: 12.5921768085

Power feature check completed.


In [26]:
redundancy_check = pd.DataFrame({"Feature": ["Voltage","Current","Power"],"Relationship": ["Raw measurement","Raw measurement","Voltage × Current"],"Scientific Meaning": ["Battery electrical potential","Charging current","Instantaneous electrical power"],
"Available_at_prediction_time": ["Yes","Yes","Yes"]})
display(redundancy_check)
print("\nPower is mathematically derived from Voltage and Current, " "but it represents a physically meaningful combined quantity.")
print("Therefore, its usefulness will be decided by model comparison ""rather than assuming it must be retained.")

,Feature,Relationship,Scientific Meaning,Available_at_prediction_time
0,Voltage,Raw measurement,Battery electrical potential,Yes
1,Current,Raw measurement,Charging current,Yes
2,Power,Voltage × Current,Instantaneous electrical power,Yes



Power is mathematically derived from Voltage and Current, but it represents a physically meaningful combined quantity.
Therefore, its usefulness will be decided by model comparison rather than assuming it must be retained.


In [27]:
print("\nSOC Features Correlation:")
display(soc_train[soc_raw_features + ["Power"]].corr().round(2))
print("\nRemaining Charging Time Features Correlation:")
display(time_train[time_raw_features + ["Power"]].corr().round(2))


SOC Features Correlation:


,Voltage,Current,Temperature,Elapsed Time,Cycle,WhAccu,Power
Voltage,1.00,0.00,0.22,0.04,NaN,0.81,0.04
Current,0.00,1.00,0.33,-0.52,NaN,0.35,1.00
Temperature,0.22,0.33,1.00,-0.36,NaN,0.43,0.34
Elapsed Time,0.04,-0.52,-0.36,1.00,NaN,-0.44,-0.52
Cycle,NaN,NaN,NaN,NaN,NaN,NaN,NaN
WhAccu,0.81,0.35,0.43,-0.44,NaN,1.00,0.37
Power,0.04,1.00,0.34,-0.52,NaN,0.37,1.00



Remaining Charging Time Features Correlation:


,Voltage,Current,Temperature,Elapsed Time,Cycle,Capacity,WhAccu,Power
Voltage,1.00,-0.29,0.07,0.06,-0.43,0.80,0.78,-0.22
Current,-0.29,1.00,0.27,-0.33,0.42,-0.28,-0.28,0.99
Temperature,0.07,0.27,1.00,-0.13,-0.21,0.24,0.24,0.27
Elapsed Time,0.06,-0.33,-0.13,1.00,-0.31,0.02,0.02,-0.33
Cycle,-0.43,0.42,-0.21,-0.31,1.00,-0.55,-0.55,0.40
Capacity,0.80,-0.28,0.24,0.02,-0.55,1.00,1.00,-0.23
WhAccu,0.78,-0.28,0.24,0.02,-0.55,1.00,1.00,-0.23
Power,-0.22,0.99,0.27,-0.33,0.40,-0.23,-0.23,1.00


In [28]:
soc_features_with_power = soc_raw_features + ["Power"]
time_features_with_power = time_raw_features + ["Power"]
print("\nSOC Feature-Target Correlation:")
soc_corr = soc_train[soc_features_with_power + ["SOC"]].corr()["SOC"].sort_values(ascending=False)
display(soc_corr.round(2))
print("\nRemaining Charging Time Feature-Target Correlation:")
time_corr = time_train[time_features_with_power + ["Remaining Charging Time"]].corr()["Remaining Charging Time"].sort_values(ascending=False)
display(time_corr.round(2))


SOC Feature-Target Correlation:


,SOC
SOC,1.00
WhAccu,0.90
Voltage,0.74
Temperature,0.41
Power,0.17
Current,0.14
Elapsed Time,-0.56
Cycle,NaN



Remaining Charging Time Feature-Target Correlation:


,Remaining Charging Time
Remaining Charging Time,1.00
Elapsed Time,0.43
Temperature,-0.13
Capacity,-0.24
WhAccu,-0.25
Voltage,-0.30
Current,-0.31
Power,-0.31
Cycle,-0.31


In [29]:
def evaluate_model(train_df,validation_df,test_df,features,target):
    X_train = train_df[features]
    X_validation = validation_df[features]
    X_test = test_df[features]
    y_train = train_df[target]
    y_validation = validation_df[target]
    y_test = test_df[target]
    model = Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler()),("model",RandomForestRegressor(n_estimators=100,random_state=42,n_jobs=-1))])
    model.fit(X_train,y_train)
    validation_prediction = model.predict(X_validation)
    test_prediction = model.predict( X_test)
    validation_mae = mean_absolute_error(y_validation,validation_prediction)
    validation_rmse = np.sqrt(mean_squared_error(y_validation,validation_prediction))
    validation_r2 = r2_score(y_validation,validation_prediction)
    test_mae = mean_absolute_error(y_test,test_prediction)
    test_rmse = np.sqrt(mean_squared_error(y_test,test_prediction))
    test_r2 = r2_score(y_test,test_prediction)
    return {"Validation MAE": validation_mae,"Validation RMSE": validation_rmse,"Validation R2": validation_r2,"Test MAE": test_mae,"Test RMSE": test_rmse,"Test R2": test_r2}
soc_raw_result = evaluate_model(soc_train,soc_validation,soc_test,soc_raw_features,"SOC")
soc_engineered_result = evaluate_model(soc_train,soc_validation,soc_test,soc_raw_features + ["Power"],"SOC")
time_raw_result = evaluate_model(time_train,time_validation,time_test,time_raw_features,"Remaining Charging Time")
time_engineered_result = evaluate_model(time_train, time_validation, time_test,time_raw_features + ["Power"],"Remaining Charging Time")
comparison_results = pd.DataFrame([
    {
        "Target": "SOC","Feature Set": "Raw",**soc_raw_result
    },

    {
        "Target": "SOC","Feature Set": "Raw + Power",**soc_engineered_result
    },

    {
        "Target": "Remaining Charging Time","Feature Set": "Raw",**time_raw_result
    },

    {
        "Target": "Remaining Charging Time","Feature Set": "Raw + Power",**time_engineered_result
    }
])
display(comparison_results)

,Target,Feature Set,Validation MAE,Validation RMSE,Validation R2,Test MAE,Test RMSE,Test R2
0,SOC,Raw,9.764684,12.055088,0.776025,0.635380,3.790385,0.991549
1,SOC,Raw + Power,9.697815,11.981020,0.778769,0.610754,3.660930,0.992116
2,Remaining Charging Time,Raw,15.699956,22.278158,0.999762,470.072511,761.071316,0.998873
3,Remaining Charging Time,Raw + Power,15.487368,22.081267,0.999766,479.950437,777.269040,0.998825


In [30]:
def compare_results(raw_result, engineered_result):
    mae_change = (engineered_result["Validation MAE"] -raw_result["Validation MAE"])
    rmse_change = (engineered_result["Validation RMSE"] -raw_result["Validation RMSE"])
    r2_change = ( engineered_result["Validation R2"] -raw_result["Validation R2"])
    return {"MAE Change": mae_change,"RMSE Change": rmse_change,"R2 Change": r2_change}
soc_improvement = compare_results(soc_raw_result,soc_engineered_result)
time_improvement = compare_results(time_raw_result,
time_engineered_result)
print("\nSOC — Raw vs Raw + Power:")
print(soc_improvement)
print("\nRemaining Charging Time — Raw vs Raw + Power:")
print(time_improvement)
print("\nInterpretation:")
print("Lower MAE/RMSE and higher R2 indicate improved validation performance.")


SOC — Raw vs Raw + Power:
{'MAE Change': -0.06686861389015775, 'RMSE Change': np.float64(-0.0740677745596745), 'R2 Change': 0.002743801750539898}

Remaining Charging Time — Raw vs Raw + Power:
{'MAE Change': -0.2125871015567018, 'RMSE Change': np.float64(-0.19689032523200112), 'R2 Change': 4.195106794724346e-06}

Interpretation:
Lower MAE/RMSE and higher R2 indicate improved validation performance.


In [31]:
if (
    soc_engineered_result["Validation RMSE"] <
    soc_raw_result["Validation RMSE"]):
    final_soc_features = soc_raw_features + ["Power"]
    soc_decision = ("Power retained because it improved validation RMSE.")
else:
    final_soc_features = soc_raw_features.copy()
    soc_decision = ("Power not retained because it did not improve validation RMSE.")
if (
    time_engineered_result["Validation RMSE"] <
    time_raw_result["Validation RMSE"]):
    final_time_features = time_raw_features + ["Power"]
    time_decision = ("Power retained because it improved validation RMSE.")
else:
    final_time_features = time_raw_features.copy()
    time_decision = ("Power not retained because it did not improve validation RMSE.")
print("\nFINAL SOC FEATURES:")
print(final_soc_features)
print("\nSOC decision:")
print(soc_decision)
print("\nFINAL REMAINING-TIME FEATURES:")
print(final_time_features)
print("\nRemaining-time decision:")
print(time_decision)


FINAL SOC FEATURES:
['Voltage', 'Current', 'Temperature', 'Elapsed Time', 'Cycle', 'WhAccu', 'Power']

SOC decision:
Power retained because it improved validation RMSE.

FINAL REMAINING-TIME FEATURES:
['Voltage', 'Current', 'Temperature', 'Elapsed Time', 'Cycle', 'Capacity', 'WhAccu', 'Power']

Remaining-time decision:
Power retained because it improved validation RMSE.


In [32]:
soc_forbidden = {"SOC","Remaining Charging Time","Final_Charging_Time","Final_Charging_Capacity"}
time_forbidden = {"SOC","Remaining Charging Time","Final_Charging_Time","Final_Charging_Capacity"}
soc_leakage = (set(final_soc_features).intersection(soc_forbidden))
time_leakage = (set(final_time_features).intersection(time_forbidden))
print("\nSOC leakage check:")
if len(soc_leakage) == 0:
    print(" No target/future-information variables detected.")
else:
    print("Possible leakage:", soc_leakage)
print("\nRemaining-time leakage check:")
if len(time_leakage) == 0:
    print("No target/future-information variables detected.")
else:
    print("Possible leakage:", time_leakage)
print("\nPower feature definition:")
print("Power = Voltage × Current")
print(
    " Power uses measurements available at prediction time."
)


SOC leakage check:
 No target/future-information variables detected.

Remaining-time leakage check:
No target/future-information variables detected.

Power feature definition:
Power = Voltage × Current
 Power uses measurements available at prediction time.


In [33]:
final_feature_list = {"SOC": final_soc_features,"Remaining Charging Time": final_time_features}
feature_list_path = os.path.join(day13_output_directory,"final_feature_list_day13.json")
with open(
    feature_list_path,
    "w"
) as file:
    json.dump(
        final_feature_list,
        file,
        indent=4
    )
print("Final feature list saved:")
print(feature_list_path)

Final feature list saved:
/content/drive/MyDrive/BATTERY_SOC_PROJECT/day13_feature_engineering/final_feature_list_day13.json


In [34]:
comparison_path = os.path.join(day13_output_directory,"raw_vs_engineered_comparison_day13.csv")
comparison_results.to_csv(comparison_path,index=False)
soc_corr_path = os.path.join(day13_output_directory,"SOC_feature_target_correlation_day13.csv")
soc_corr.to_csv(soc_corr_path,header=True)
time_corr_path = os.path.join(day13_output_directory,"remaining_time_feature_target_correlation_day13.csv")
time_corr.to_csv(
    time_corr_path,
    header=True
)
print("\nFiles saved successfully:")
print("\n1. Raw vs Engineered Comparison:")
print(comparison_path)
print("\n2. SOC Feature-Target Correlation:")
print(soc_corr_path)
print("\n3. Remaining Charging Time Feature-Target Correlation:")
print(time_corr_path)


Files saved successfully:

1. Raw vs Engineered Comparison:
/content/drive/MyDrive/BATTERY_SOC_PROJECT/day13_feature_engineering/raw_vs_engineered_comparison_day13.csv

2. SOC Feature-Target Correlation:
/content/drive/MyDrive/BATTERY_SOC_PROJECT/day13_feature_engineering/SOC_feature_target_correlation_day13.csv

3. Remaining Charging Time Feature-Target Correlation:
/content/drive/MyDrive/BATTERY_SOC_PROJECT/day13_feature_engineering/remaining_time_feature_target_correlation_day13.csv


In [35]:
soc_train.to_csv(
    os.path.join(day13_output_directory,"SOC_train_engineered_day13.csv"),
    index=False)
soc_validation.to_csv(
    os.path.join(day13_output_directory,"SOC_validation_engineered_day13.csv"),
    index=False)
soc_test.to_csv(
    os.path.join(
        day13_output_directory,
        "SOC_test_engineered_day13.csv"),
    index=False)
time_train.to_csv(
    os.path.join(day13_output_directory,"remaining_time_train_engineered_day13.csv"),
    index=False)
time_validation.to_csv(
    os.path.join(day13_output_directory,"remaining_time_validation_engineered_day13.csv"),
    index=False)
time_test.to_csv(
    os.path.join(day13_output_directory,"remaining_time_test_engineered_day13.csv"),
    index=False)
print("Engineered datasets saved.")

Engineered datasets saved.


In [36]:
print("\nFeature engineering that is done:")
print(" Power = Voltage × Current")
print("\nRedundancy:")
print(" Voltage, Current and Power relationship reviewed")
print("\nMulticollinearity:")
print("\nRaw vs engineered:")
print(" Model performance comparison completed")
print("\nLeakage:")
print(" Final feature lists checked for target/future information")
print("\nFinal SOC features:")
print(final_soc_features)
print("\nFinal Remaining Charging Time features:")
print(final_time_features)
print("\nOutputs saved to:")
print(day13_output_directory)


Feature engineering that is done:
 Power = Voltage × Current

Redundancy:
 Voltage, Current and Power relationship reviewed

Multicollinearity:

Raw vs engineered:
 Model performance comparison completed

Leakage:
 Final feature lists checked for target/future information

Final SOC features:
['Voltage', 'Current', 'Temperature', 'Elapsed Time', 'Cycle', 'WhAccu', 'Power']

Final Remaining Charging Time features:
['Voltage', 'Current', 'Temperature', 'Elapsed Time', 'Cycle', 'Capacity', 'WhAccu', 'Power']

Outputs saved to:
/content/drive/MyDrive/BATTERY_SOC_PROJECT/day13_feature_engineering
